In [33]:
import numpy as np

In [10]:
import pandas as pd 

df=pd.read_csv(
    "/Users/vaibhav/Desktop/AI_project/intelligent-crop-recommendation/ml-engine/datasets/processed/groundwater/groundwater_cleaned.csv"
)
df["Monitoring Date"] = pd.to_datetime(
    df["Monitoring Date"]
)

df["Station Code"] = df["Station Code"].astype("string")

df["GP Code"] = df["GP Code"].astype("Int64").astype("string")

df["Basin Code"] = df["Basin Code"].astype("Int64").astype("string")

df["Sub Basin Code"] = df["Sub Basin Code"].astype("Int64").astype("string")

#sort the data station and  date  
df=df.sort_values(by=["Station Code","Monitoring Date"]).reset_index(drop=True)
#verifying the result 
print(df.shape)

print("\nData types:")
print(df.dtypes)

print("\nFirst 10 observations:")
print(
    df[
        ["Station Code", "Monitoring Date", "Ground Water Level"]
    ].head(10)
)



(124384, 23)

Data types:
_id                                    int64
Station Code                          string
State                                    str
State Code                             int64
District                                 str
District Code                          int64
Block                                    str
Block Code                             int64
GP Name                                  str
GP Code                               string
Basin                                    str
Basin Code                            string
Sub Basin                                str
Sub Basin Code                        string
Latitude                             float64
Longitude                            float64
Monitoring Date               datetime64[us]
Ground Water Level                   float64
negative_level_flag                     bool
large_monthly_change_flag               bool
shared_reading_review_flag              bool
missing_coordinate_flag      

In [9]:
date_order = (
    df.groupby("Station Code")["Monitoring Date"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print("All stations chronologically sorted:", date_order.all())

All stations chronologically sorted: True


create the copy

In [11]:
df_copy=df.copy()

inserting year colunms

In [ ]:
df_copy["Year"]=df_copy["Monitoring Date"].dt.year

In [23]:
print(df_copy.columns)
print(df_copy["Year"].head())

Index(['_id', 'Station Code', 'State', 'State Code', 'District',
       'District Code', 'Block', 'Block Code', 'GP Name', 'GP Code', 'Basin',
       'Basin Code', 'Sub Basin', 'Sub Basin Code', 'Latitude', 'Longitude',
       'Monitoring Date', 'Ground Water Level', 'negative_level_flag',
       'large_monthly_change_flag', 'shared_reading_review_flag',
       'missing_coordinate_flag', 'suspicious_coordinate_flag', 'Year',
       'Month'],
      dtype='str')
0    2021
1    2021
2    2021
3    2021
4    2021
Name: Year, dtype: int32


inserting months

In [21]:
df_copy["Month"]=df_copy["Monitoring Date"].dt.month

In [22]:
print(df_copy.columns)
print(df_copy["Month"].head())

Index(['_id', 'Station Code', 'State', 'State Code', 'District',
       'District Code', 'Block', 'Block Code', 'GP Name', 'GP Code', 'Basin',
       'Basin Code', 'Sub Basin', 'Sub Basin Code', 'Latitude', 'Longitude',
       'Monitoring Date', 'Ground Water Level', 'negative_level_flag',
       'large_monthly_change_flag', 'shared_reading_review_flag',
       'missing_coordinate_flag', 'suspicious_coordinate_flag', 'Year',
       'Month'],
      dtype='str')
0    1
1    2
2    3
3    4
4    5
Name: Month, dtype: int32


creating Quarter columns 

In [28]:
df_copy["Quarter"]=df_copy["Monitoring Date"].dt.quarter

In [29]:
print(df_copy.columns)
print(df_copy["Quarter"].head())

Index(['_id', 'Station Code', 'State', 'State Code', 'District',
       'District Code', 'Block', 'Block Code', 'GP Name', 'GP Code', 'Basin',
       'Basin Code', 'Sub Basin', 'Sub Basin Code', 'Latitude', 'Longitude',
       'Monitoring Date', 'Ground Water Level', 'negative_level_flag',
       'large_monthly_change_flag', 'shared_reading_review_flag',
       'missing_coordinate_flag', 'suspicious_coordinate_flag', 'Year',
       'Month', 'Quarter'],
      dtype='str')
0    1
1    1
2    1
3    2
4    2
Name: Quarter, dtype: int32


In [30]:
print(df_copy[["Monitoring Date", "Year", "Month", "Quarter"]].head(10))

  Monitoring Date  Year  Month  Quarter
0      2021-01-01  2021      1        1
1      2021-02-01  2021      2        1
2      2021-03-01  2021      3        1
3      2021-04-01  2021      4        2
4      2021-05-01  2021      5        2
5      2021-06-01  2021      6        2
6      2021-07-01  2021      7        3
7      2021-08-01  2021      8        3
8      2021-09-01  2021      9        3
9      2021-10-01  2021     10        4


creating season columns

In [34]:

condition = [
    (df_copy["Month"] >= 1) & (df_copy["Month"] <= 2),
    (df_copy["Month"] >= 3) & (df_copy["Month"] <= 5),
    (df_copy["Month"] >= 6) & (df_copy["Month"] <= 9),
    (df_copy["Month"] >= 10) & (df_copy["Month"] <= 12)
]

choice = [
    "Winter",
    "Pre-Monsoon",
    "Monsoon",
    "Post-Monsoon"
]

df_copy["Season"] = np.select(
    condlist=condition,
    choicelist=choice,
    default=None
)

In [40]:
print(df_copy["Season"].value_counts(dropna=False))
print(df_copy["Season"].isnull().sum())
summary=(df_copy.groupby(by="Month")
         ["Season"].unique().sort_index)
print(summary)

Season
Monsoon         41787
Pre-Monsoon     31077
Post-Monsoon    30958
Winter          20562
Name: count, dtype: int64
0
<bound method Series.sort_index of Month
1           [Winter]
2           [Winter]
3      [Pre-Monsoon]
4      [Pre-Monsoon]
5      [Pre-Monsoon]
6          [Monsoon]
7          [Monsoon]
8          [Monsoon]
9          [Monsoon]
10    [Post-Monsoon]
11    [Post-Monsoon]
12    [Post-Monsoon]
Name: Season, dtype: object>


In [39]:
print(df_copy["Season"].dtypes)

str


In [41]:
summary=df_copy.groupby(by="Station Code").agg(
    Number_of_Observation=("Ground Water Level","count")
)

In [42]:
print(summary)

              Number_of_Observation
Station Code                       
010801D                          58
020109b                          60
020116a                          60
020203A                          60
020204C                          24
...                             ...
91816                            24
91817                            24
CGWB1                            12
CGWB2                            12
CGWB3                            12

[2709 rows x 1 columns]


In [45]:
print("Exactly 60 : ",(summary["Number_of_Observation"]==60).sum())
print("Fewer than 60 : ",(summary["Number_of_Observation"]<60).sum())
print("More than 60 :",(summary["Number_of_Observation"]>60).sum())
print("Min",summary["Number_of_Observation"].min())
print("Max",summary["Number_of_Observation"].max())
print("Max",summary["Number_of_Observation"].mean())

Exactly 60 :  1100
Fewer than 60 :  1609
More than 60 : 0
Min 12
Max 60
Max 45.915097822074564
